In [3]:
from pathlib import Path
import re, json
from typing import Any, Dict, List, Optional
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.docstore.document import Document

INPUT_DIR   = Path("./parsed_data")
SPLITTER = RecursiveCharacterTextSplitter(
    chunk_size=1200, chunk_overlap=200,
    separators=["\n## ", "\n### ", "\n#### ", "\n\n", "\n"],
)
RE_YEAR = re.compile(r"(19|20)\d{2}")
CONTENT_KEYS = ("page_content", "document", "text", "content")

def _to_int_year(v: Any) -> Optional[int]:
    if v is None: return None
    if isinstance(v, int): return v
    if isinstance(v, str):
        m = RE_YEAR.search(v)
        if m: return int(m.group(0))
    return None

def _infer_year_from_text(text: str) -> Optional[int]:
    m = RE_YEAR.search(text)
    return int(m.group(0)) if m else None

def _sanitize_metadata(meta: Dict[str, Any]) -> Dict[str, Any]:
    """Chroma 1.x 친화 스칼라화: year=int, list/tuple→' | ' 또는 JSON, dict→JSON"""
    clean: Dict[str, Any] = {}
    meta = dict(meta or {})
    if "year" in meta:
        iv = _to_int_year(meta.get("year"))
        if iv is not None:
            clean["year"] = iv
    for k, v in meta.items():
        if k == "year": 
            continue
        if isinstance(v, (str, int, float, bool)) or v is None:
            clean[k] = v
        elif isinstance(v, (list, tuple)):
            if all(isinstance(x, (str, int, float, bool)) or x is None for x in v):
                clean[k] = " | ".join("" if x is None else str(x) for x in v)
            else:
                clean[k] = json.dumps(v, ensure_ascii=False)
        elif isinstance(v, dict):
            clean[k] = json.dumps(v, ensure_ascii=False)
        else:
            clean[k] = str(v)
    return clean

def _augment_content(text: str, meta: Dict[str, Any]) -> str:
    """e5 권장 'passage:' 프리픽스 + 라벨로 검색 품질 향상"""
    parts = []
    if meta.get("company"): parts.append(f"회사:{meta['company']}")
    if meta.get("year") is not None: parts.append(f"연도:{meta['year']}")
    if meta.get("section"): parts.append(f"섹션:{meta['section']}")
    if meta.get("source"): parts.append(f"소스:{meta['source']}")
    if meta.get("statement"): parts.append(f"표:{meta['statement']}")
    if meta.get("line_item"): parts.append(f"항목:{meta['line_item']}")
    label = " ".join(parts)
    return f"passage: {label} || {text}".strip()

def _pick_text(rec: Dict[str, Any]) -> str:
    for k in CONTENT_KEYS:
        v = rec.get(k)
        if isinstance(v, str) and v.strip():
            return v.strip()
    return ""


In [5]:
def docs_from_md(path: Path) -> List[Document]:
    raw = path.read_text(encoding="utf-8")
    year = _to_int_year(path.name) or _to_int_year(raw) or _infer_year_from_text(raw)
    meta = {
        "year": year,
        "file_type": "md",
        "filename": path.name,
        "company": "삼성전자",
        "source": f"배당_{year}.pdf" if year else f"{path.stem}.pdf",
        "section": f"배당정보-{year}" if year else "배당정보",
    }
    meta = _sanitize_metadata(meta)
    content = _augment_content(raw, meta)
    base = Document(page_content=content, metadata=meta)
    return SPLITTER.split_documents([base])

md_files = sorted(INPUT_DIR.glob("*.md"))
print("MD files:", len(md_files))
if md_files:
    _sample = docs_from_md(md_files[0])
    print("MD sample chunk meta:", _sample[0].metadata)
    print("MD sample chunk text:", _sample[0].page_content[:160])

MD files: 11
MD sample chunk meta: {'year': 2014, 'file_type': 'md', 'filename': '배당_2014.md', 'company': '삼성전자', 'source': '배당_2014.pdf', 'section': '배당정보-2014'}
MD sample chunk text: passage: 회사:삼성전자 연도:2014 섹션:배당정보-2014 소스:배당_2014.pdf || ## 배당 정보 - 2014년
| 구 분                    | 주식의 종류   | 2014년   | 2013년   | 2012년   |
|:-----------------


In [ ]:
def docs_from_jsonl(path: Path) -> List[Document]:
    docs: List[Document] = []
    with path.open("r", encoding="utf-8") as f:
        for ln, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except Exception as e:
                print(f"[WARN] JSON parse fail {path.name}:{ln} - {e}")
                continue
            text = _pick_text(rec)
            if not text:
                continue
            meta = rec.get("metadata", {}) or {}
            meta.setdefault("company", "삼성전자")
            if "year" not in meta:
                y = _to_int_year(text)
                if y is not None:
                    meta["year"] = y
            meta.setdefault("file_type", "jsonl")
            meta.setdefault("filename", path.name)
            meta = _sanitize_metadata(meta)
            content = _augment_content(text, meta)
            docs.append(Document(page_content=content, metadata=meta))
    return SPLITTER.split_documents(docs)

jsonl_files = sorted(INPUT_DIR.glob("*.jsonl"))
print("JSONL files:", len(jsonl_files))
if jsonl_files:
    _sample = docs_from_jsonl(jsonl_files[0])
    print("JSONL sample chunk meta:", _sample[0].metadata)
    print("JSONL sample chunk text:", _sample[0].page_content[:160])

JSONL files: 17
JSONL sample chunk meta: {'year': 2014, 'company': '삼성전자', 'company_anchor': '삼성전자', 'year_anchor': '삼성전자-2014', 'section': 'SEC1', 'kind': 'auditor', 'group_id': 'sec1-2014-root', 'parent_id': 'sec1-2014-root', 'source': '감사보고서_2014_preprocess.html#SEC1', 'file_type': 'jsonl', 'filename': 'sec1_auditor_kam_all.jsonl'}
JSONL sample chunk text: passage: 회사:삼성전자 연도:2014 섹션:SEC1 소스:감사보고서_2014_preprocess.html#SEC1 || 2014년 삼성전자 재무제표의 외부감사는 삼일회계법인이 수행했습니다.


In [7]:
all_docs: List[Document] = []
for p in md_files:
    all_docs.extend(docs_from_md(p))
for p in jsonl_files:
    all_docs.extend(docs_from_jsonl(p))

print("총 청크 수:", len(all_docs))
assert all_docs, "로드된 문서가 없습니다."

총 청크 수: 1406


In [8]:
B = 512
for i in range(0, len(all_docs), B):
    db.add_documents(all_docs[i:i+B])
    print(f"add {i} ~ {min(i+B, len(all_docs))} / {len(all_docs)}")

print("✅ 임베딩 & 저장 완료:", PERSIST_DIR.as_posix())

add 0 ~ 512 / 1406
add 512 ~ 1024 / 1406
add 1024 ~ 1406 / 1406
✅ 임베딩 & 저장 완료: chroma_db


In [ ]:
# 간단 검색
q = "2019년 배당 현금 관련 내용"   # 예시
hits = db.similarity_search(f"query: {q}", k=3, filter={"year": 2019})
for i, d in enumerate(hits, 1):
    print(f"\n[Hit {i}] year={d.metadata.get('year')} file={d.metadata.get('filename')}")
    print(d.page_content)


[Hit 1] year=2019 file=sec2_현금흐름표.jsonl
passage: 회사:삼성전자 연도:2019 섹션:SEC2 표:현금흐름표 항목:배당금의지급 || 2019년 삼성전자의 현금흐름/재무활동현금흐름 중 배당금의지급은(는) -9,618,210 백만원입니다.

[Hit 2] year=2019 file=sec2_현금흐름표.jsonl
passage: 회사:삼성전자 연도:2019 섹션:SEC2 표:현금흐름표 항목:배당금수입 || 2019년 삼성전자의 현금흐름/영업활동현금흐름 중 배당금수입은(는) 4,625,181 백만원입니다.

[Hit 3] year=2019 file=배당_2019.md
passage: 회사:삼성전자 연도:2019 섹션:배당정보-2019 소스:배당_2019.pdf || ## 배당 정보 - 2019년
| 구 분                    | 주식의 종류   | 2019년   | 2018년   | 2017년   |
|:-------------------------|:--------------|:---------|:---------|:---------|
| 주당액면가액(원)         |               | 100      | 100      | 100      |
| (연결)당기순이익(백만원) |               | 21505054 | 43890877 | 41344569 |
| (별도)당기순이익(백만원) |               | 15353323 | 32815127 | 28800837 |
| (연결)주당순이익(원)     |               | 3166     | 6461     | 5997     |
| 현금배당금총액(백만원)   |               | 9619243  | 9619243  | 5826302  |
| 주식배당금총액(백만원)   |               |          |          |          |
| (연결)현금배당성향(%)    |          

In [ ]:
# === RAG: 많이 가져오고(큰 k) → 압축 → LLM ===
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import EmbeddingsFilter
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

# 1) Retriever (큰 k + 점수 컷)
retriever = db.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": 300,               # 많이 긁기 (원하면 500까지 올려도 OK)
        "score_threshold": 0.10 # 낮출수록 더 많이 남김
        # "filter": {"year": {"$in": [2014, 2019]}}  # 연도 필터 필요 시
    },
)

# 2) 압축(2차 컷): 임베딩 기반 필터
compressor = EmbeddingsFilter(
    embeddings=emb,
    similarity_threshold=0.5,   # ↑ 올리면 더 타이트(잡음 감소, 문서 수 감소)
)
comp_retriever = ContextualCompressionRetriever(
    base_retriever=retriever,
    base_compressor=compressor,
)

# 3) LLM + 프롬프트
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2,api_key='')  # OPENAI_API_KEY는 환경변수로

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "너는 삼성전자 재무/사업보고서 QA 전문가다. "
     "주어진 문서 컨텍스트만 근거로 수치·연도를 명확히 답하라. "
     "모순되면 이유를 설명하고, 출처(연도/파일)를 함께 덧붙여라."),
    ("human", "질문: {input}\n\n컨텍스트:\n{context}")
])

doc_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=prompt,
    document_variable_name="context",  # prompt의 {context}와 매칭
)

# 4) Retrieval → LLM 결합
rag_chain = create_retrieval_chain(
    retriever=comp_retriever,
    combine_docs_chain=doc_chain,      # 주의: combine_docs_chain (0.3.x)
)

# 5) 실행 헬퍼
def run_rag(question: str, show_sources: bool = True):
    result = rag_chain.invoke({"input": question})
    answer = result["answer"]
    ctx_docs = result.get("context", [])
    print("=== 답변 ===\n", answer)
    print("\n=== 사용된 문서 개수 ===", len(ctx_docs))
    if show_sources:
        for i, d in enumerate(ctx_docs, 1):
            m = d.metadata or {}
            print(f"[{i}] year={m.get('year')} file={m.get('filename')} section={m.get('section')}")
    return result

# 예시
_ = run_rag("2014년과 2019년 배당금을 비교해줘")


=== 답변 ===
 2014년과 2019년의 삼성전자 배당금을 비교하면 다음과 같습니다:

### 2014년
- **주당 현금배당금 (원)**: 
  - 보통주: 20,000 원
  - 우선주: 20,050 원
- **현금배당금총액 (백만원)**: 2,999,972 백만원
- **현금배당성향 (%)**: 13.0%
- **현금배당수익률 (%)**: 
  - 보통주: 1.5%
  - 우선주: 1.9%

### 2019년
- **주당 현금배당금 (원)**: 
  - 보통주: 1,416 원
  - 우선주: 1,417 원
- **현금배당금총액 (백만원)**: 9,619,243 백만원
- **현금배당성향 (%)**: 44.7%
- **현금배당수익률 (%)**: 
  - 보통주: 2.6%
  - 우선주: 3.1%

### 비교 요약
- **주당 현금배당금**: 2014년 보통주 20,000 원에서 2019년 1,416 원으로 감소.
- **현금배당금총액**: 2014년 2,999,972 백만원에서 2019년 9,619,243 백만원으로 증가.
- **현금배당성향**: 2014년 13.0%에서 2019년 44.7%로 증가.
- **현금배당수익률**: 보통주가 2014년 1.5%에서 2019년 2.6%로 증가.

이러한 변화는 삼성전자의 수익성 및 배당 정책의 변화에 기인할 수 있습니다. 

출처: 
- 2014년 배당 정보: 배당_2014.pdf
- 2019년 배당 정보: 배당_2019.pdf

=== 사용된 문서 개수 === 20
[1] year=2014 file=배당_2014.md section=배당정보-2014
[2] year=2014 file=sec2_현금흐름표.jsonl section=SEC2
[3] year=2019 file=배당_2019.md section=배당정보-2019
[4] year=2014 file=sec2_자본변동표.jsonl section=SEC2
[5] year=2014 file=sec2_현금흐름표.jsonl section=SEC2
[6] ye

In [ ]:
# === Analyst-grade Writer (fixed) ===
import re
from langchain.schema import Document
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain

# 0) LLM (길게 작성 허용)
writer = ChatOpenAI(
    model="gpt-4o",
    temperature=0.2,
    max_tokens=1600,   # 필요시 더 늘려도 OK
    api_key="" # secret
)

# 1) 문서 포맷 (출처/연도/파일/섹션 노출)
document_prompt = PromptTemplate.from_template(
    "[source:{source} | year:{year} | file:{filename} | section:{section}]\n"
    "{page_content}"
)

# 2) 리포트 프롬프트 (애널리스트 톤)
report_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "너는 삼성전자 재무/사업보고서 전문 애널리스트다. "
     "반드시 제공된 컨텍스트(문서들)만 근거로 사실을 서술하고, 수치·연도·단위를 명확히 써라. "
     "추정이 필요하면 근거와 불확실성을 함께 기술하고, 결론에 인용한 문서의 year/filename을 각 섹션 말미에 괄호로 표기해라."),
    ("human",
     "분석 주제: {input}\n\n"
     "아래 컨텍스트를 바탕으로 애널리스트 보고서를 작성하라. 한국어로 쓰고, Markdown을 사용하라.\n\n"
     "요구사항:\n"
     "1) **Executive Summary**: 3~6줄로 핵심 결론과 숫자 하이라이트.\n"
     "2) **데이터 표(Table)**: 비교 연도별 주요 지표(주당배당금, 현금배당총액, 배당성향, 배당수익률 등)를 표로 정리. 단위 명확히.\n"
     "3) **변화 분석(Drivers)**: 정책 변화(배당 정책/자사주/특별배당/정책 공지), 영업/일회성 요인 분해.\n"
     "4) **해석/함의(Implications)**: 투자자 관점의 의미, 지속가능성 평가.\n"
     "5) **리스크 & 감안사항**: 데이터 공시 범위, 회계/정책 변경, 환율/가격 사이클 등.\n"
     "6) **출처 명시**: 각 섹션 말미에 (year, filename) 형태로 주요 인용 출처 1~2개만 표기.\n"
     "형식 가이드: 700~1000단어 권장, 불릿과 소제목을 적절히 사용.\n\n"
     "컨텍스트 문서들:\n{context}")
])

# 3) 문서-주입(stuff) 체인
doc_chain = create_stuff_documents_chain(
    llm=writer,
    prompt=report_prompt,
    document_variable_name="context",
    document_prompt=document_prompt,
)

# 4) 메타 결측 보정 유틸
YEAR_RE = re.compile(r"(19|20)\d{2}")

def _infer_year_from_text(text: str):
    m = YEAR_RE.search(text or "")
    return int(m.group(0)) if m else None

def normalize_docs(docs):
    """document_prompt에서 요구하는 메타키 ['filename','section','source','year']를 보장"""
    normed = []
    for d in docs:
        m = dict(d.metadata or {})
        m.setdefault("filename", m.get("filename") or "unknown")
        m.setdefault("section",  m.get("section")  or "N/A")
        m.setdefault("source",   m.get("source")   or m.get("filename") or m.get("section") or "N/A")
        # year가 없으면 텍스트/파일명에서 추정
        if m.get("year") is None:
            y = _infer_year_from_text(d.page_content) or _infer_year_from_text(m.get("filename",""))
            if y is not None:
                m["year"] = y
        normed.append(Document(page_content=d.page_content, metadata=m))
    return normed

# 5) 실행 함수
def run_analyst_report(question: str, show_sources: bool = True):
    # comp_retriever는 이전 단계에서 생성된 객체를 사용합니다.
    raw_docs = comp_retriever.get_relevant_documents(question)
    docs = normalize_docs(raw_docs)

    # doc_chain.invoke(...)는 문자열을 반환합니다.
    answer_text = doc_chain.invoke({"input": question, "context": docs})

    print("=== 애널리스트 리포트 ===\n")
    print(answer_text)

    if show_sources:
        print("\n--- 사용된 문서(요약) ---")
        for i, d in enumerate(docs, 1):
            m = d.metadata or {}
            print(f"[{i}] year={m.get('year')} file={m.get('filename')} source={m.get('source')} section={m.get('section')}")
    return {"answer": answer_text, "context": docs}

# 사용 예시
_ = run_analyst_report("2014년부터 2024년까지 삼성전자 사업부문의 변화를 보여줘")


=== 애널리스트 리포트 ===

# 삼성전자 사업부문 변화 분석 (2014-2024)

## Executive Summary
삼성전자는 2014년부터 2024년까지 다양한 사업부문에서 지속적인 변화를 겪었습니다. 2024년 삼성전자의 연결기준 매출은 301조원, 영업이익은 33조원으로 기록되었으며, 이는 2014년의 매출 206조원, 영업이익 25조원과 비교하여 상당한 성장을 보여줍니다. 특히, DX 부문과 DS 부문에서의 기술 혁신과 제품 포트폴리오 확장이 주요 성장 동력으로 작용하였습니다. 그러나, 글로벌 경제 불확실성과 경쟁 심화는 여전히 도전 과제로 남아 있습니다. (2024, 경영진의견_2024.overview.jsonl)

## 데이터 표(Table)

| 연도 | 주당배당금(원) | 현금배당총액(억원) | 배당성향(%) | 배당수익률(%) |
|------|----------------|-------------------|-------------|---------------|
| 2014 | 1,000          | 3,000             | 20          | 1.5           |
| 2022 | 1,500          | 4,500             | 25          | 2.0           |
| 2024 | 2,000          | 6,000             | 30          | 2.5           |

*단위: 주당배당금(원), 현금배당총액(억원), 배당성향(%), 배당수익률(%)

## 변화 분석(Drivers)

### 정책 변화
- **배당 정책**: 삼성전자는 배당금과 배당성향을 지속적으로 증가시켜 주주 가치를 강화하고 있습니다. 2014년 주당배당금은 1,000원이었으나, 2024년에는 2,000원으로 증가하였습니다.
- **자사주 매입 및 특별배당**: 자사주 매입과 특별배당을 통해 주주 환원 정책을 강화하고 있으며, 이는 주가 안정화에 기여하고 있습니다.

